In [1]:
# =========================================================
# External SHAP Analysis for CNN Ensemble (Background = 50)
# =========================================================
# Purpose:
# - Perform SHAP analysis on trained CNN ensemble
# - Background samples: 50 randomly selected samples
# - External dataset: CRLM (dat_crlm.csv)
# - Designed for Reviewer #2 (interpretability)
# =========================================================

import os
import random
import numpy as np
import pandas as pd
import tensorflow as tf
import shap
import joblib
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler

e:\conda-envs\tensorflow\lib\site-packages\scipy\__init__.py:146: UserWarning: A NumPy version >=1.16.5 and <1.23.0 is required for this version of SciPy (detected version 1.24.4
  warnings.warn(f"A NumPy version >={np_minversion} and <{np_maxversion}"
IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html


In [ ]:
# =========================================================
# Step 0: Global settings
# =========================================================
SEED = 42
os.environ["PYTHONHASHSEED"] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

print("Random seed fixed to 42")

In [ ]:
# =========================================================
# Step 1: File paths (STRICTLY as provided)
# =========================================================
print("\n--- Step 1: Configure file paths ---")

BASE_DIR = "D:/结直肠癌肝转移Biomarker 诊断/新的策略/Autoencoder"

# Cross-validation model output paths
CV_OUTPUT_DIR = "D:/temp_output_cv"
ENSEMBLE_MODEL_DIR = os.path.join(CV_OUTPUT_DIR, "ensemble_models")

# Validation data path
VALIDATION_DATA_DIR = os.path.join(BASE_DIR, "validation_datasets")

# Results output path
OUTPUT_DIR = os.path.join(BASE_DIR, "crlm_validation_with_cv_model")
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Input file paths
CV_LABEL_ENCODER_FILE = os.path.join(ENSEMBLE_MODEL_DIR, "label_encoder.pkl")
CV_GENES_FILE = os.path.join(CV_OUTPUT_DIR, "used_functional_genes_cv.txt")
CRLM_DATA_FILE = os.path.join(VALIDATION_DATA_DIR, "dat_crlm.csv")

print(f"Ensemble model directory: {ENSEMBLE_MODEL_DIR}")
print(f"CRLM validation data: {CRLM_DATA_FILE}")
print(f"Results output directory: {OUTPUT_DIR}")

In [ ]:
# =========================================================
# Step 2: Load functional genes
# =========================================================
with open(CV_GENES_FILE, "r") as f:
    model_genes = [g.strip() for g in f if g.strip()]

print(f"Loaded {len(model_genes)} functional genes")

In [ ]:
# =========================================================
# Step 3: Load external CRLM dataset
# =========================================================
crlm = pd.read_csv(CRLM_DATA_FILE, index_col=0)

# Gold-standard label mapping (same as original external script)
y_true = crlm["status"].apply(
    lambda x: 1 if "metastasis" in str(x).lower() else 0
).values

print("External label distribution:")
print(pd.Series(y_true).value_counts())

# Build feature matrix in the SAME gene order as training
X_ext = pd.DataFrame(index=crlm.index, columns=model_genes)

available = [g for g in model_genes if g in crlm.columns]
missing = [g for g in model_genes if g not in crlm.columns]

X_ext[available] = crlm[available]
X_ext[missing] = 0.0

print(f"External samples: {X_ext.shape[0]}")
print(f"Missing genes filled with zero: {len(missing)}")

In [ ]:
# =========================================================
# Step 4: Load CNN ensemble models and scalers
# =========================================================
def load_cnn_ensemble():
    models, scalers = [], []
    for i in range(1, 6):
        model_path = os.path.join(ENSEMBLE_MODEL_DIR, f"model_fold{i}.keras")
        scaler_path = os.path.join(ENSEMBLE_MODEL_DIR, f"scaler_fold{i}.pkl")

        models.append(tf.keras.models.load_model(model_path))
        scalers.append(joblib.load(scaler_path))

        print(f"Loaded CNN fold {i}")
    return models, scalers

cnn_models, cnn_scalers = load_cnn_ensemble()

In [ ]:
# =========================================================
# Step 5: Prepare background samples (n = 50)
# =========================================================
print("\n--- Step 5: Prepare SHAP background (n=50) ---")

n_background = min(50, X_ext.shape[0])
background_idx = np.random.choice(X_ext.index, n_background, replace=False)

X_background = X_ext.loc[background_idx]
print(f"Background samples selected: {X_background.shape[0]}")

In [ ]:
# =========================================================
# Step 6: SHAP analysis (per fold, then average)
# =========================================================
print("\n--- Step 6: Compute SHAP values for CNN ensemble ---")

shap_values_all_folds = []

for i, (model, scaler) in enumerate(zip(cnn_models, cnn_scalers), 1):
    print(f"Running SHAP for CNN fold {i}")

    # Scale background and external data using fold-specific scaler
    X_bg_scaled = scaler.transform(X_background)
    X_bg_cnn = np.expand_dims(X_bg_scaled, axis=-1)

    X_ext_scaled = scaler.transform(X_ext)
    X_ext_cnn = np.expand_dims(X_ext_scaled, axis=-1)

    # GradientExplainer is recommended for TF/Keras CNN
    explainer = shap.GradientExplainer(model, X_bg_cnn)

    shap_values = explainer.shap_values(X_ext_cnn)[0]  # (N, genes, 1)
    shap_values = shap_values.squeeze(-1)              # (N, genes)

    shap_values_all_folds.append(shap_values)

# Average SHAP values across 5 folds
shap_values_mean = np.mean(np.stack(shap_values_all_folds, axis=0), axis=0)

print("SHAP values computed and averaged across folds")

In [ ]:
# =========================================================
# Step 7: Save SHAP results
# =========================================================
print("\n--- Step 7: Save SHAP outputs ---")

# 1) Save numeric SHAP values
mean_abs_shap = np.mean(np.abs(shap_values_mean), axis=0)

shap_df = pd.DataFrame({
    "Gene": model_genes,
    "MeanAbsSHAP": mean_abs_shap
}).sort_values("MeanAbsSHAP", ascending=False)

shap_table_file = os.path.join(OUTPUT_DIR, "SHAP_mean_abs_values_CNN_external.csv")
shap_df.to_csv(shap_table_file, index=False)
print(f"SHAP table saved: {shap_table_file}")

# 2) SHAP summary plot (beeswarm)
plt.figure(figsize=(10, 6))
shap.summary_plot(
    shap_values_mean,
    X_ext,
    feature_names=model_genes,
    max_display=30,
    show=False
)
plt.tight_layout()
beeswarm_file = os.path.join(OUTPUT_DIR, "SHAP_beeswarm_top30_CNN_external.png")
plt.savefig(beeswarm_file, dpi=300)
plt.close()
print(f"SHAP beeswarm plot saved: {beeswarm_file}")

# 3) SHAP bar plot
plt.figure(figsize=(8, 6))
shap.summary_plot(
    shap_values_mean,
    X_ext,
    feature_names=model_genes,
    plot_type="bar",
    max_display=30,
    show=False
)
plt.tight_layout()
barplot_file = os.path.join(OUTPUT_DIR, "SHAP_bar_top30_CNN_external.png")
plt.savefig(barplot_file, dpi=300)
plt.close()
print(f"SHAP bar plot saved: {barplot_file}")

print("External SHAP analysis completed successfully.")
print(f"All outputs saved to: {OUTPUT_DIR}")